# Datamine ST1GX Process Example

This notebook demonstrates how to configure and run the **`st1gx`** process wrapper in `dmstudio`.

## Process Description

## ST1GX

Calculates general summary statistics for a single field, in log or non-log mode with or without class selection using Sturgess' Rule.

The calculated statistics include:-

* total number of records in the file.
* number of values used for statistical analysis.
* number of missing values.
* minimum, maximum, range, total and mean.
* variance, standard deviation and standard error.
* skewness and kurtosis.
* coefficient of variation.
* mean plus 2 and mean plus 4 standard deviations.
* median.
* mode.

These results are displayed in the Output control bar, and can be copied to the printer or printfile. A list of **BIN** sizes with **BIN** counts and percentage frequency for each class, cumulative frequency, class interval and number of **BINS** are also calculated. In addition, printer plots on a normal probability scale are calculated and can be sent to the printer or printfile.

The first bin in the histogram plot contains all values up to **MINIMUM**. The last bin contains all values above the top value. If **LOGMODE** is set to 1, antilogs are automatically calculated.

Values of skewness and kurtosis calculated are interpreted as:

**SKEWNESS** |  = 0. No distortion (Gaussian).
---|---
|  > 0\. Positive skew (to the right).
|  < 0\. Negative skew (to the left).
**KURTOSIS** |  = 0. Mesokurtic (Gaussian).
|  > 0\. Leptokurtic (peaked).
|  < 0\. Platikurtic (flat).

### Input Files:

* **in** (Table):
  Input file.
  Required=Yes

### Output Files:

### Fields:

* **value** (Numeric : IN):
  Field for statistics.
  Default=Undefined
  Required=Yes

### Parameters:

* **binsize**:
  Bin width for histogram [computed].
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **minimum**:
  Minimum bin limit [computed].
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **sturgess**:

* **Options** (1: class interval [bin width] and minimum class value [bin limit] are):
  calculated according to the following equation: Class interval = 1 + 3.3 log N [Sturgess
  Rule] and values within each bin are placed at the mid-point of each class (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **logmode**:

* **Options** (1: log transformation [base 10] is calculated (0).):
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **perc**:

* **Options** (0: numeric bin count for histogram; >0 percentage bin count for histogram (0).):
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('st1gx')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute st1gx
print("Running st1gx...")
dm_cmd.st1gx(
    in_i='t_assays',  # required
    value_f='optional',  # required
    # binsize_p='optional',  # optional
    # minimum_p='optional',  # optional
    # sturgess_p=0,  # optional
    # logmode_p=0,  # optional
    # perc_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("st1gx execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_st1gx_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")